# 08 — Daily Signal and Paper Trading

Bu notebook:

1. Güncel BIST100 ve XU100 verilerini indirir.
2. Robot kalite kontrollerini ve indikatörlerini uygular.
3. Final stratejiyle günlük alış/satış planı üretir.
4. Paper-trading portföyünü JSON dosyasında saklar.
5. Emirlerin gerçek veya varsayımsal dolumlarını ayrı hücrelerle kaydeder.

Sinyaller kapanış verisiyle oluşur. `BUY_NEXT_OPEN` ve `SELL_NEXT_OPEN`
emirleri bir sonraki işlem günü açılışında uygulanmalıdır.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import DataConfig
from src.data_loader import (
    load_bist_tickers,
    download_robot_bundle,
)
from src.data_quality import run_quality_pipeline
from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.paper_trading import (
    load_paper_state,
    save_paper_state,
    positions_dataframe,
    calculate_fill_position_size,
    record_buy,
    record_sell,
    append_closed_trade,
    mark_to_market,
    append_equity_snapshot,
)
from src.daily_signal import (
    create_daily_plan,
    save_daily_plan,
)


## 1. Güncel veriyi indir


In [ ]:
# Yaklaşık 2,5 yıllık veri EMA200 ve diğer indikatörler için yeterlidir.
DOWNLOAD_START = (
    pd.Timestamp.today().normalize()
    - pd.Timedelta(days=900)
).strftime("%Y-%m-%d")

data_config = DataConfig(
    start=DOWNLOAD_START,
    end=None,
    auto_adjust=True,
    yfinance_repair=False,
)

tickers = load_bist_tickers(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bist100_sirketler.xlsx"
)

raw_stocks, raw_market, download_errors = (
    download_robot_bundle(
        tickers=tickers,
        config=data_config,
    )
)

print("Hisse satırı:", len(raw_stocks))
print("Endeks satırı:", len(raw_market))
print("İndirme hatası:", len(download_errors))

if not download_errors.empty:
    display(download_errors)


## 2. Kalite kontrolü ve özellik mühendisliği


In [ ]:
stock_quality = run_quality_pipeline(
    raw_stocks,
    config=data_config,
    apply_split_repairs=True,
)

market_quality = run_quality_pipeline(
    raw_market,
    config=data_config,
    apply_split_repairs=True,
)

stock_features = add_indicators(
    stock_quality.clean
)

market_features = add_indicators(
    market_quality.clean
)

market_regime = build_market_regime(
    market_features
)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=True,
)

print("Özellikli hisse satırı:", len(scored_prices))
print("Hisse sayısı:", scored_prices["Ticker"].nunique())
print(
    "Son özellik tarihi:",
    scored_prices["Date"].max(),
)


## 3. Paper-trading state dosyasını yükle


In [ ]:
PAPER_DIR = (
    PROJECT_ROOT
    / "results"
    / "paper_trading"
)

STATE_PATH = PAPER_DIR / "state.json"
TRADES_PATH = PAPER_DIR / "closed_trades.csv"
EQUITY_PATH = PAPER_DIR / "equity_history.csv"
PLANS_DIR = PAPER_DIR / "daily_plans"

paper_state = load_paper_state(
    path=STATE_PATH,
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
)

display(positions_dataframe(paper_state))

print("Nakit:", paper_state["cash"])
print("Açık pozisyon:", len(paper_state["positions"]))


## 4. Günlük işlem planını oluştur


In [ ]:
daily_plan = create_daily_plan(
    scored_prices=scored_prices,
    state=paper_state,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    signal_date=None,
    minimum_coverage_ratio=0.60,
)

display(daily_plan.summary)

print("SATIŞ / STOP PLANLARI")
display(daily_plan.sell_orders)

print("ALIŞ PLANLARI")
display(daily_plan.buy_orders)

print("MEVCUT POZİSYONLAR")
display(daily_plan.hold_positions)


In [ ]:
plan_paths = save_daily_plan(
    plan=daily_plan,
    output_directory=PLANS_DIR,
)

print("Plan dosyaları:")
for name, path in plan_paths.items():
    print(name, "→", path)


## 5. Portföyü güncel fiyatlarla değerle

Bu hücre state dosyasını değiştirmez; yalnızca mevcut portföy değerini hesaplar.


In [ ]:
latest_rows = (
    scored_prices.loc[
        scored_prices["Date"].eq(
            daily_plan.signal_date
        )
    ]
    .drop_duplicates("Ticker", keep="last")
)

latest_price_map = dict(
    zip(
        latest_rows["Ticker"],
        latest_rows["Close"],
    )
)

snapshot = mark_to_market(
    state=paper_state,
    latest_prices=latest_price_map,
)

snapshot["Signal_Date"] = (
    daily_plan.signal_date.isoformat()
)

display(pd.DataFrame([snapshot]))

append_equity_snapshot(
    snapshot=snapshot,
    path=EQUITY_PATH,
)


## 6. Alış gerçekleştiğinde kaydet

Aşağıdaki örneği doğrudan çalıştırma. Emir gerçekten dolduktan sonra:

- `TICKER`
- `FILL_PRICE`
- `FILL_DATE`
- ilgili plan satırı

değerlerini gir.

Lot, gerçekleşen fiyat üzerinden yeniden hesaplanır.


In [ ]:
# ÖRNEK — değerleri düzenlemeden çalıştırma.
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-07-27"
# FILL_PRICE = 70.25
#
# plan_row = (
#     daily_plan.buy_orders
#     .loc[
#         daily_plan.buy_orders["Ticker"].eq(TICKER)
#     ]
#     .iloc[0]
# )
#
# current_snapshot = mark_to_market(
#     paper_state,
#     latest_price_map,
# )
#
# actual_stop = (
#     FILL_PRICE
#     - FINAL_STRATEGY_CONFIG.initial_stop_atr
#     * float(plan_row["ATR"])
# )
#
# actual_shares = calculate_fill_position_size(
#     state=paper_state,
#     fill_price=FILL_PRICE,
#     stop_loss=actual_stop,
#     current_equity=current_snapshot["Equity"],
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
# )
#
# record_buy(
#     state=paper_state,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     shares=actual_shares,
#     stop_loss=actual_stop,
#     signal_score=int(plan_row["Score"]),
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
# )
#
# save_paper_state(
#     paper_state,
#     STATE_PATH,
# )
#
# display(positions_dataframe(paper_state))


## 7. Satış gerçekleştiğinde kaydet

Satış fiyatını gerçekleşen fiyatla gir. Fonksiyon nakdi günceller,
pozisyonu kapatır ve işlem kaydını CSV günlüğüne ekleyebilir.


In [ ]:
# ÖRNEK — değerleri düzenlemeden çalıştırma.
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-08-10"
# FILL_PRICE = 76.80
# EXIT_REASON = "Trailing Stop"
#
# paper_state, closed_trade = record_sell(
#     state=paper_state,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     reason=EXIT_REASON,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
# )
#
# save_paper_state(
#     paper_state,
#     STATE_PATH,
# )
#
# append_closed_trade(
#     trade=closed_trade,
#     path=TRADES_PATH,
# )
#
# display(pd.DataFrame([closed_trade]))
# display(positions_dataframe(paper_state))


## Günlük kullanım sırası

1. Piyasa kapandıktan sonra 1–5. bölümleri çalıştır.
2. `BUY_NEXT_OPEN` ve `SELL_NEXT_OPEN` emirlerini kontrol et.
3. Ertesi gün gerçekleşen dolumları 6 veya 7. bölümde kaydet.
4. State dosyasını GitHub'a gönderme; `.gitignore` içinde tut.
